In [2]:
import pandas as pd
import time
import os
from collections import deque
import urllib.request, urllib.error, urllib.parse
import json
from dotenv import load_dotenv

REST_URL = "http://data.bioontology.org"

# Create a .env file with your API key
# Format: BIO_PORTAL_API_KEY=your_api_key_here
load_dotenv()
API_KEY = os.environ.get("BIO_PORTAL_API_KEY", "")


# --- CONFIGURATION ---
# Output file name
OUTPUT_FILE = "doid_disease_tree.csv"

# Batch size for saving progress (save every N nodes processed)
SAVE_INTERVAL = 50 

# Rate limiting (seconds between API calls)
API_DELAY = 0.1 

def get_json(url):
    opener = urllib.request.build_opener()
    opener.addheaders = [('Authorization', 'apikey token=' + API_KEY)]
    return json.loads(opener.open(url).read())


In [3]:
# Access the DOID ontology specifically
doid_acronym = "DOID"  # Human Disease Ontology
doid_url = f"{REST_URL}/ontologies/{doid_acronym}"
doid_ontology = get_json(doid_url)
print(f"Accessing DOID: {doid_ontology['name']}")

# Get the root classes of the DOID ontology
roots_url = doid_ontology['links']['roots']
roots = get_json(roots_url)
print("\nRoot classes in DOID:")
if len(roots) > 1:
    for root in roots:
        if root['prefLabel'] == 'disease':
            disease_root = root
            print("\tFOUND ROOT:\n", root)
        

Accessing DOID: Human Disease Ontology

Root classes in DOID:
	FOUND ROOT:
 {'prefLabel': 'disease', 'synonym': [], 'definition': ['A disease is a disposition (i) to undergo pathological processes that (ii) exists in an organism because of one or more disorders in that organism.'], 'cui': [], 'semanticType': [], 'obsolete': False, 'created': None, 'modified': None, 'memberOf': [], 'inScheme': [], '@id': 'http://purl.obolibrary.org/obo/DOID_4', '@type': 'http://www.w3.org/2002/07/owl#Class', 'links': {'self': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4', 'ontology': 'https://data.bioontology.org/ontologies/DOID', 'children': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/children', 'parents': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/parents', 'descendants': 'https://data.bioontology.org/ontologies/DOID/classes/http%

In [4]:
disease_root

{'prefLabel': 'disease',
 'synonym': [],
 'definition': ['A disease is a disposition (i) to undergo pathological processes that (ii) exists in an organism because of one or more disorders in that organism.'],
 'cui': [],
 'semanticType': [],
 'obsolete': False,
 'created': None,
 'modified': None,
 'memberOf': [],
 'inScheme': [],
 '@id': 'http://purl.obolibrary.org/obo/DOID_4',
 '@type': 'http://www.w3.org/2002/07/owl#Class',
 'links': {'self': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4',
  'ontology': 'https://data.bioontology.org/ontologies/DOID',
  'children': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/children',
  'parents': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/parents',
  'descendants': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/descendants',

In [ ]:
import pandas as pd
import time
import os
from collections import deque
from datetime import datetime

# --- CONFIGURATION ---
OUTPUT_FILE = "doid_disease_tree.csv"
SAVE_INTERVAL = 10  # Save every 10 nodes (was 50 - too infrequent!)
API_DELAY = 0.05  # Reduced from 0.1 to speed up (still rate-limited)

def get_json_with_retry(url, retries=3):
    """Fetch JSON with basic retry logic and rate limiting"""
    for attempt in range(retries):
        try:
            time.sleep(API_DELAY)
            return get_json(url)
        except Exception as e:
            if attempt == retries - 1:
                print(f"❌ Failed to fetch {url} after {retries} attempts: {e}")
                return None
            print(f"⚠️ Error fetching {url}, retrying ({attempt+1}/{retries})...")
            time.sleep(1 * (attempt + 1))
    return None

def get_all_children_pagination(children_url):
    """Fetch all children handling BioPortal pagination via 'nextPage' link"""
    all_children = []
    
    page = get_json_with_retry(children_url)
    if not page:
        return []
        
    if 'collection' in page:
        all_children.extend(page['collection'])
    
    next_page_url = page.get("links", {}).get("nextPage")
    page_count = 1
    
    while next_page_url:
        page_count += 1
        page = get_json_with_retry(next_page_url)
        if not page:
            break
            
        if 'collection' in page:
            all_children.extend(page['collection'])
        
        next_page_url = page.get("links", {}).get("nextPage")
    
    return all_children

def build_disease_tree(root_node):
    """
    Breadth-First Search (BFS) traversal to build the disease tree.
    """
    
    queue = deque([ (root_node, "disease", 0) ])
    visited_ids = set([root_node['@id']])
    database_rows = []
    nodes_processed = 0
    start_time = time.time()
    
    print(f"🚀 Starting traversal from root: {root_node['prefLabel']}")
    print(f"📂 Output will be saved to: {OUTPUT_FILE}")
    print(f"💾 Checkpoint interval: Every {SAVE_INTERVAL} nodes")
    print("-" * 60)
    
    try:
        while queue:
            current_node, current_path, current_level = queue.popleft()
            nodes_processed += 1
            
            # Extract data
            node_id = current_node.get('@id', '')
            node_label = current_node.get('prefLabel', 'Unknown')
            
            row = {
                'id': node_id,
                'level': current_level,
                'path': current_path,
                'prefLabel': node_label
            }
            database_rows.append(row)
            
            # --- ENHANCED LOGGING ---
            elapsed = time.time() - start_time
            if nodes_processed % 5 == 0:  # Log every 5 nodes
                rate = nodes_processed / elapsed if elapsed > 0 else 0
                print(f"[{nodes_processed:4d}] Level {current_level} | Queue: {len(queue):4d} | "
                      f"Processing: {node_label[:40]:40s} | "
                      f"Rate: {rate:.2f} nodes/sec | Elapsed: {elapsed:.0f}s")
            
            # --- SAVE PROGRESS (MORE FREQUENT) ---
            if nodes_processed % SAVE_INTERVAL == 0:
                df_partial = pd.DataFrame(database_rows)
                df_partial.to_csv(OUTPUT_FILE, index=False)
                elapsed = time.time() - start_time
                rate = nodes_processed / elapsed if elapsed > 0 else 0
                print(f"💾 CHECKPOINT: Saved {nodes_processed} rows | "
                      f"Rate: {rate:.2f} nodes/sec | Elapsed: {elapsed:.0f}s")
                print(f"   Current queue size: {len(queue)} | Current level: {current_level}")

            # --- FETCH CHILDREN ---
            links = current_node.get('links', {})
            children_url = links.get('children')
            
            if children_url:
                # Show we're fetching children
                if nodes_processed % 10 == 0:
                    print(f"   ↳ Fetching children for: {node_label[:40]}")
                
                children = get_all_children_pagination(children_url)
                
                if children:
                    if nodes_processed % 10 == 0:
                        print(f"   ↳ Found {len(children)} children")
                
                for child in children:
                    child_id = child.get('@id')
                    child_label = child.get('prefLabel')
                    
                    if child_id and child_id not in visited_ids:
                        visited_ids.add(child_id)
                        new_path = f"{current_path}/{child_label}"
                        queue.append((child, new_path, current_level + 1))
            
    except KeyboardInterrupt:
        print("\n🛑 Traversal interrupted by user!")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        print("\n📝 Saving final data...")
        df_final = pd.DataFrame(database_rows)
        df_final.to_csv(OUTPUT_FILE, index=False)
        total_time = time.time() - start_time
        print(f"✅ DONE. Saved {len(df_final)} rows to {OUTPUT_FILE}")
        print(f"⏱️  Total time: {total_time:.0f} seconds ({total_time/60:.1f} minutes)")
        return df_final

# --- EXECUTION ---
if 'disease_root' in locals() and disease_root:
    root_self_url = disease_root['links']['self']
    full_disease_root = get_json_with_retry(root_self_url)
    
    if full_disease_root:
        df = build_disease_tree(full_disease_root)
        print("\n--- Preview ---")
        print(df.head(20))
    else:
        print("❌ Could not fetch full disease root details.")
else:
    print("❌ 'disease_root' variable not found.")

🚀 Starting traversal from root: disease
📂 Output will be saved to: doid_disease_tree.csv
💾 Checkpoint interval: Every 10 nodes
------------------------------------------------------------
[   5] Level 1 | Queue:  252 | Processing: interstitial lung disease                | Rate: 0.42 nodes/sec | Elapsed: 12s
[  10] Level 1 | Queue:  364 | Processing: external ear disease                     | Rate: 0.46 nodes/sec | Elapsed: 22s
💾 CHECKPOINT: Saved 10 rows | Rate: 0.46 nodes/sec | Elapsed: 22s
   Current queue size: 364 | Current level: 1
   ↳ Fetching children for: external ear disease
   ↳ Found 3 children
[  15] Level 1 | Queue:  396 | Processing: third cranial nerve disease              | Rate: 0.56 nodes/sec | Elapsed: 27s
[  20] Level 1 | Queue:  413 | Processing: vein disease                             | Rate: 0.63 nodes/sec | Elapsed: 32s
💾 CHECKPOINT: Saved 20 rows | Rate: 0.63 nodes/sec | Elapsed: 32s
   Current queue size: 413 | Current level: 1
   ↳ Fetching children for: v

# Improved Version of BFS 